# Output Parser
# -------------

# To format the output in a structured way



In [ ]:
from google import genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import JsonOutputParser
from dotenv import load_dotenv
import os,warnings
from pydantic import BaseModel, Field

In [ ]:
warnings.filterwarnings("ignore")

env = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\bosch\2026\5_advancedPE\code\config.env"
if load_dotenv(env):
    gemini_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=gemini_key)

In [ ]:
# Step 2: Wrap the Gemini API call as a LangChain Runnable
def call_gemini(prompt):
    # PromptTemplate produces a PromptValue object
    prompt_text = prompt.to_string()

    response = client.models.generate_content(model="gemini-3.1-flash-lite", contents=prompt_text)

    # Return an AIMessage for StrOutputParser
    return AIMessage(content=response.text)

gemini_runnable = RunnableLambda(call_gemini)

In [ ]:
# 1) String Parser
parser = StrOutputParser()

prompt = ChatPromptTemplate.from_template("""
You are an expert in creating technical contents.

Write a 200-word article on the topic: {title}.
""")

# form the chain by combining the prompt, gemini_runnable, and parser

chain1 = prompt | gemini_runnable
chain2 = prompt | gemini_runnable | parser

In [ ]:
r1 = chain1.invoke({"title": "Small Scale Industries"})
r2 = chain2.invoke({"title": "Small Scale Industries"})

In [ ]:
print(r1.content)

In [ ]:
print(r2)

In [ ]:
# 2) JSON Parser
# Parse the output as a JSON format

In [ ]:
# Step 1: Define the JSON output parser
parser = JsonOutputParser()

format_instructions = parser.get_format_instructions()

# Step 2: Create the prompt template
prompt = ChatPromptTemplate.from_template("""
You are an AI assistant that classifies support tickets.

Extract the following fields:
- category
- priority
- suggested_action

{format_instructions}

User Query:
{query}
""")

In [ ]:
# Step 3: Initialize the Gemini LLM
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite",api_key=gemini_key,temperature=0)

# Step 4: Create the chain
chain = prompt | llm | parser

In [ ]:
# Step 5: Invoke the chain
response = chain.invoke({
    "query": "My laptop is overheating and shutting down frequently.",
    "format_instructions": format_instructions
})

In [ ]:
print(response)

# --------------------------
# Pydantic Structured Output
# --------------------------

In [ ]:
# Step 1: Define the expected output structure
class Person(BaseModel):
    name: str = Field(description="Person's name")
    age: int = Field(description="Person's age")
    profession: str = Field(description="Person's occupation")

# Step 2: Initialize the Gemini model
# llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", api_key=gemini_key, temperature=0)

# Step 3: Add the structured-output schema
structured_llm = llm.with_structured_output(Person)

# Step 4: Provide the input
input_text = """
Extract the person's name, age, and profession from the following text:

John is 35 years old and works as a Data Scientist.
"""

In [ ]:
# Step 5: Invoke Gemini
response = structured_llm.invoke(input_text)

In [ ]:
# Step 6: Display the result
print(response)
(response.name, response.age, response.profession)

# 4) Comma Separated List Output Parser

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate

parser = CommaSeparatedListOutputParser()

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite",api_key=gemini_key,temperature=0)

# Step 4: Create the chain
chain = prompt | llm | parser

prompt = PromptTemplate.from_template("""
List five popular sports in India.

{format_instructions}
""")

chain = prompt | llm | parser

In [ ]:
response = chain.invoke({"format_instructions": parser.get_format_instructions()})

In [ ]:
print(response)